# Baseline Notebook — PEM vs VWAP MR pada M5 dan M15

Notebook ini memakai fungsi load/resample yang kamu berikan, lalu menguji:
- **PEM** versi research-refactor dengan structure stop dan fixed-R target
- **VWAP MR** versi research-refactor dengan stop dibekukan dari entry price

Asumsi konservatif:
- entry = next bar open
- intrabar ambiguity = SL first
- single-position-only
- baseline ini belum memasukkan spread/slippage/commission


In [ ]:
import json
from pathlib import Path
import pandas as pd
from xauusd_pem_vwap_research import *

## 1) Set path CSV M1

In [ ]:
DATA_PATH = 'xauusd_m1.csv'  # ganti sesuai file kamu

## 2) Load + resample ke M5 dan M15

In [ ]:
df_m1 = load_ohlcv(DATA_PATH)
df_m5 = resample_ohlcv(df_m1, '5min')
df_m15 = resample_ohlcv(df_m1, '15min')
print('Rows M1 :', len(df_m1))
print('Rows M5 :', len(df_m5))
print('Rows M15:', len(df_m15))
df_m5.head()

## 3) Jalankan baseline backtest

In [ ]:
pem_m5 = backtest_pem(df_m5, timeframe='M5', rr_target=2.0, timeout_bars=24)
pem_m15 = backtest_pem(df_m15, timeframe='M15', rr_target=2.0, timeout_bars=12)
vwap_m5 = backtest_vwap_mr(df_m5, timeframe='M5', timeout_bars=20, stop_loss_pct=0.005)
vwap_m15 = backtest_vwap_mr(df_m15, timeframe='M15', timeout_bars=10, stop_loss_pct=0.005)

## 4) Ringkasan metrik

In [ ]:
metrics = {
    'PEM_M5': summarize_trades(pem_m5),
    'PEM_M15': summarize_trades(pem_m15),
    'VWAP_MR_M5': summarize_trades(vwap_m5),
    'VWAP_MR_M15': summarize_trades(vwap_m15),
}
metrics_df = pd.DataFrame(metrics).T
metrics_df

## 5) Sample trades

In [ ]:
pem_m5.head(), vwap_m5.head()

## 6) Plot equity in R

In [ ]:
plot_equity(pem_m5, 'PEM M5 Equity in R')
plot_equity(pem_m15, 'PEM M15 Equity in R')
plot_equity(vwap_m5, 'VWAP MR M5 Equity in R')
plot_equity(vwap_m15, 'VWAP MR M15 Equity in R')

## 7) Simpan output

In [ ]:
OUT = Path('outputs_baseline')
OUT.mkdir(exist_ok=True)
pem_m5.to_csv(OUT / 'pem_m5_trades.csv', index=False)
pem_m15.to_csv(OUT / 'pem_m15_trades.csv', index=False)
vwap_m5.to_csv(OUT / 'vwap_m5_trades.csv', index=False)
vwap_m15.to_csv(OUT / 'vwap_m15_trades.csv', index=False)
metrics_df.to_csv(OUT / 'metrics_summary.csv')
with open(OUT / 'metrics_summary.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('saved to', OUT)

## 8) Catatan

- **PEM** di sini adalah baseline research, bukan copy persis TradingView default, karena script asli default `useSL=false`.
- **VWAP MR** di sini membekukan stop dari **entry price**, bukan dari `close` tiap bar.
- Tahap berikutnya yang tetap perlu: spread/slippage stress, session breakdown, regime filter, walk-forward.
